In [3]:
from collections import Counter
from pathlib import Path
import pickle
import automated_llm_probes as alp

TARGET_N = 700
NAMES = ['claude-haiku-4.5',
 'claude-opus-4.5',
 'claude-opus-4.7',
 'claude-opus-5',
 'claude-sonnet-4.5',
 'gpt-3.5-turbo',
 'gpt-4-turbo',
 'gpt-4-turbo',
 'gpt-4o',
 'gpt-4o-mini',
 'gpt-5.4',
 'grok-4.2',
 'grok-4.3',
 'grok-4.5',
 'grok-build-0.1',
 'llama-3.1-8b',
 'llama-3.2-3b',
 'llama-4-guard-12b',
 'llama-4-maverick',
 'llama-4-scout']

HUMAN_N = {  # 3,083 human CWT rows; shares scale with TARGET_N
    "stamp letter send": 558,
    "superpower": 261,
    "2305": 101,            # human file: "title (2305 or Execution)" = 202
    "execution": 101,
    "belief faith sing": 153,
    "gloom payment exist": 153,
    "organ empire comply": 153,
    "petrol diesel pump": 153,
    "statement stealth detect": 153,
    "year week embark": 153,
    "frame": 147,
    "glow": 141,
    "death": 86,
    "delay": 86,
    "enemy": 86,
    "illness": 86,
    "lie": 86,
    "marriage": 86,
    "joy": 85,
    "shade": 85,
    "simplicity": 85,
    "sky": 85,
}

def targets(n):
    tot = sum(HUMAN_N.values())
    raw = {c: n * k / tot for c, k in HUMAN_N.items()}
    out = {c: int(v) for c, v in raw.items()}
    for c in sorted(raw, key=lambda c: raw[c] - out[c], reverse=True):
        if sum(out.values()) >= n:
            break
        out[c] += 1
    return out

tgt = targets(TARGET_N)
models = [m for m in alp.ready_models() if m["name"] in NAMES]
print("N", TARGET_N, [m["name"] for m in models])

for m in models:
    have = Counter()
    for p in Path("cwt", m["name"]).rglob("*.pickle"):
        try:
            row = pickle.load(open(p, "rb"))
        except Exception:
            continue
        cue = (row.get("kwargs") or {}).get("cue") or row.get("cue")
        if isinstance(cue, (list, tuple)):
            cue = " ".join(str(x) for x in cue)
        if cue:
            have[str(cue).strip().lower()] += 1
    n_have = sum(have.values())
    print(f"\n{m['name']}  {n_have}")
    for cue, want in tgt.items():
        need = max(0, want - have.get(cue, 0))
        print(f"  {cue:28s} {have.get(cue, 0):4d}/{want:<3d}  {'ok' if need == 0 else f'+{need}'}")
        if need:
            alp.collect("CWT", models=[m], n_per_model=n_have + need, cue=cue.split())
            n_have += need

N 700 ['grok-4.2', 'grok-4.3', 'grok-4.5', 'grok-build-0.1', 'gpt-3.5-turbo', 'gpt-4-turbo', 'gpt-4o', 'gpt-5.4', 'gpt-4o-mini', 'gpt-4-turbo', 'llama-4-guard-12b', 'llama-4-scout', 'llama-4-maverick', 'llama-3.2-3b', 'llama-3.1-8b', 'claude-sonnet-4.5', 'claude-haiku-4.5', 'claude-opus-4.5', 'claude-opus-4.7', 'claude-opus-5']

grok-4.2  0
  stamp letter send               0/127  +127
  grok-4.2: 720/127 done — skip
  superpower                      0/59   +59
  grok-4.2: 720/186 done — skip
  2305                            0/23   +23
  grok-4.2: 720/209 done — skip
  execution                       0/23   +23
  grok-4.2: 720/232 done — skip
  belief faith sing               0/35   +35
  grok-4.2: 720/267 done — skip
  gloom payment exist             0/35   +35
  grok-4.2: 720/302 done — skip
  organ empire comply             0/35   +35
  grok-4.2: 720/337 done — skip
  petrol diesel pump              0/35   +35
  grok-4.2: 720/372 done — skip
  statement stealth detect        0/35  

  gpt-4-turbo: 720/337 done — skip
  petrol diesel pump              0/35   +35
  gpt-4-turbo: 720/372 done — skip
  statement stealth detect        0/35   +35
  gpt-4-turbo: 720/407 done — skip
  year week embark                0/35   +35
  gpt-4-turbo: 720/442 done — skip
  frame                           0/33   +33
  gpt-4-turbo: 720/475 done — skip
  glow                            0/32   +32
  gpt-4-turbo: 720/507 done — skip
  death                           0/20   +20
  gpt-4-turbo: 720/527 done — skip
  delay                           0/20   +20
  gpt-4-turbo: 720/547 done — skip
  enemy                           0/20   +20
  gpt-4-turbo: 720/567 done — skip
  illness                         0/19   +19
  gpt-4-turbo: 720/586 done — skip
  lie                             0/19   +19
  gpt-4-turbo: 720/605 done — skip
  marriage                        0/19   +19
  gpt-4-turbo: 720/624 done — skip
  joy                             0/19   +19
  gpt-4-turbo: 720/643 done — skip
  sha

  llama-4-guard-12b: 600/527 done — skip
  delay                           0/20   +20
  llama-4-guard-12b: 600/547 done — skip
  enemy                           0/20   +20
  llama-4-guard-12b: 600/567 done — skip
  illness                         0/19   +19
  llama-4-guard-12b: 600/586 done — skip
  lie                             0/19   +19
  llama-4-guard-12b: 600 collected, 5 to collect


CWT: 100%|█████████████████████████████████████████████████████████████████| 5/5 [00:05<00:00,  1.06s/it]


  marriage                        0/19   +19
  llama-4-guard-12b: 605 collected, 19 to collect


...

CWT: 100%|███████████████████████████████████████████████████████████████| 19/19 [02:14<00:00,  7.07s/it]


  simplicity                      0/19   +19
  claude-opus-5: 662 collected, 19 to collect


CWT: 100%|███████████████████████████████████████████████████████████████| 19/19 [02:05<00:00,  6.59s/it]


  sky                             0/19   +19
  claude-opus-5: 681 collected, 19 to collect


CWT: 100%|███████████████████████████████████████████████████████████████| 19/19 [02:09<00:00,  6.81s/it]
